In [1]:
# !pip install scikit-learn==1.5.1


## Connecting Google Drive

In [7]:
from google.colab import drive

drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [8]:
# checking scikit-learn version
import sklearn
print(sklearn.__version__)

1.6.0


## Importing Dependencies

In [9]:
#Installation of required libraries
import numpy as np
import pandas as pd
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
# import sklearn version 1.5.1
import sklearn
print(sklearn.__version__)
print(np.__version__)
print(pd.__version__)
from sklearn.preprocessing import scale, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix, accuracy_score, mean_squared_error, r2_score, roc_auc_score, roc_curve, classification_report, recall_score, precision_score, f1_score, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import KFold
import warnings
warnings.simplefilter(action = "ignore")

1.6.0
1.26.4
2.2.2


## EDA

In [6]:
#Reading the dataset
df = pd.read_csv("/content/drive/MyDrive/Diabetes_Prediction/diabetes_2k.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Diabetes_Prediction/diabetes_2k.csv'

In [ ]:
# The first 5 observation units of the data set were accessed.
df.head(20)

In [ ]:
# The size of the data set was examined. It consists of 768 observation units and 9 variables.
df.shape, df.columns

In [ ]:
#Feature information
df.info()

In [ ]:
# Descriptive statistics of the data set accessed.
df.describe([0.10,0.25,0.50,0.75,0.90,0.95,0.99]).T

In [ ]:
# The distribution of the Outcome variable was examined.
df["Outcome"].value_counts()*100/len(df)

In [ ]:
# The classes of the outcome variable were examined.
df.Outcome.value_counts()

In [ ]:
# The histagram of the Age variable was reached.
df["Age"].hist(edgecolor = "black");

In [ ]:
print("Max Age: " + str(df["Age"].max()) + " Min Age: " + str(df["Age"].min()))

In [ ]:
# Histogram and density graphs of all variables were accessed.
fig, ax = plt.subplots(4,2, figsize=(16,16))
sns.distplot(df.Age, bins = 20, ax=ax[0,0])
sns.distplot(df.Pregnancies, bins = 20, ax=ax[0,1])
sns.distplot(df.Glucose, bins = 20, ax=ax[1,0])
sns.distplot(df.BloodPressure, bins = 20, ax=ax[1,1])
sns.distplot(df.SkinThickness, bins = 20, ax=ax[2,0])
sns.distplot(df.Insulin, bins = 20, ax=ax[2,1])
sns.distplot(df.DiabetesPedigreeFunction, bins = 20, ax=ax[3,0])
sns.distplot(df.BMI, bins = 20, ax=ax[3,1])

In [ ]:
df.groupby("Outcome").agg({"Pregnancies":"mean"})

In [ ]:
df.groupby("Outcome").agg({"Age":"mean"})

In [ ]:
df.groupby("Outcome").agg({"Age":"max"})

In [ ]:
f, ax = plt.subplots(1, 2, figsize=(18, 8), dpi=300)

# Pie chart
df['Outcome'].value_counts().plot.pie(explode=[0, 0.1], autopct='%1.1f%%', ax=ax[0], shadow=True)
ax[0].set_title('Target')
ax[0].set_ylabel('')
ax[0].legend(labels=['Non-Diabetic (0)', 'Diabetic (1)'], loc='upper left')

# Countplot
sns.countplot(x='Outcome', data=df, palette=['g', 'r'], ax=ax[1])
ax[1].set_title('Outcome')
ax[1].legend(labels=['Non-Diabetic (0)', 'Diabetic (1)'], loc='upper right')

plt.show()


In [ ]:
# Access to the correlation of the data set was provided. What kind of relationship is examined between the variables.
# If the correlation value is> 0, there is a positive correlation. While the value of one variable increases, the value of the other variable also increases.
# Correlation = 0 means no correlation.
# If the correlation is <0, there is a negative correlation. While one variable increases, the other variable decreases.
# When the correlations are examined, there are 2 variables that act as a positive correlation to the Salary dependent variable.
# These variables are Glucose. As these increase, Outcome variable increases.
df.corr()

In [ ]:

""" Pregnancies	Glucose	BloodPressure	SkinThickness	Insulin	BMI	DiabetesPedigreeFunction	Age	Outcome
Pregnancies	1.000000	0.129459	0.141282	-0.081672	-0.073535	0.017683	-0.033523	0.544341	0.221898
Glucose	0.129459	1.000000	0.152590	0.057328	0.331357	0.221071	0.137337	0.263514	0.466581
BloodPressure	0.141282	0.152590	1.000000	0.207371	0.088933	0.281805	0.041265	0.239528	0.065068
SkinThickness	-0.081672	0.057328	0.207371	1.000000	0.436783	0.392573	0.183928	-0.113970	0.074752
Insulin	-0.073535	0.331357	0.088933	0.436783	1.000000	0.197859	0.185071	-0.042163	0.130548
BMI	0.017683	0.221071	0.281805	0.392573	0.197859	1.000000	0.140647	0.036242	0.292695
DiabetesPedigreeFunction	-0.033523	0.137337	0.041265	0.183928	0.185071	0.140647	1.000000	0.033561	0.173844
Age	0.544341	0.263514	0.239528	-0.113970	-0.042163	0.036242	0.033561	1.000000	0.238356
Outcome	0.221898	0.466581	0.065068	0.074752	0.130548	0.292695	0.173844	0.238356	1.000000 """
# Correlation matrix graph of the data set
f, ax = plt.subplots(figsize= [12,12])
sns.heatmap(df.corr(), annot=True, fmt=".2f", ax=ax, cmap = "magma" )
ax.set_title("Correlation Matrix", fontsize=20)
plt.savefig("output.jpg")
plt.show()


## Data Preprocessing

 Missing Observation Analysis¶
We saw on df.head() that some features contain 0, it doesn't make sense here and this indicates missing value Below we replace 0 value by NaN:

In [ ]:
df[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']] = df[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']].replace(0,np.NaN)

In [ ]:
df.head(30)

In [ ]:
# Now, we can look at where are missing values
df.isnull().sum()

In [ ]:
# Have been visualized using the missingno library for the visualization of missing observations.
# Plotting
import missingno as msno
msno.bar(df);

In [ ]:
# The missing values ​​will be filled with the median values ​​of each variable.
def median_target(var):
    temp = df[df[var].notnull()]
    temp = temp[[var, 'Outcome']].groupby(['Outcome'])[[var]].median().reset_index()
    return temp

In [ ]:
# The values to be given for incomplete observations are given the median value of people who are not sick and the median values of people who are sick.
columns = df.columns
columns = columns.drop("Outcome")
for i in columns:
    median_target(i)
    df.loc[(df['Outcome'] == 0 ) & (df[i].isnull()), i] = median_target(i)[i][0]
    df.loc[(df['Outcome'] == 1 ) & (df[i].isnull()), i] = median_target(i)[i][1]

In [ ]:
df.head()

In [ ]:
# Missing values were filled.
df.isnull().sum()

## Visualization

In [ ]:
# correlation heatmap
# sns.heatmap(df.corr(), annot=True)

In [ ]:
# plotting scatter matrix
# sns.pairplot( df )

In [ ]:
# plotting scatter matrix
# sns.pairplot( df, hue="Outcome" )

## Splitting traning and test data

In [ ]:
x = df.drop("Outcome", axis=1)
y = df["Outcome"]

In [ ]:
x

In [ ]:
y

In [ ]:
# train test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

In [ ]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
# printing the splitted data

print(x_train)
print(x_test)
print(y_train)
print(y_test)

## Feature Scaling

In [ ]:
import pickle
scaler = StandardScaler()
scaler.fit(x_train)

# Save the scaler to a file
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)
x_train_scaled, x_test_scaled

In [ ]:
x_test_scaled

In [ ]:
# import pickle
# from sklearn.preprocessing import StandardScaler

# # Assuming x_train is your training data array
# x_train = [
#     [1, 85, 66, 29, 0, 26.6, 0.351, 31],
#     [8, 183, 64, 0, 0, 23.3, 0.672, 32],
#     [1, 89, 66, 23, 94, 28.1, 0.167, 21],
#     # ... more training data
# ]

# scaler = StandardScaler()
# scaler.fit(x_train)

# # Save the scaler to a file
# with open('scaler.pkl', 'wb') as f:
#     pickle.dump(scaler, f)

# # Transform x_train for checking
# x_train_scaled = scaler.transform(x_train)
# print(x_train_scaled)


In [ ]:
# # transforming a data with pickle model
# scaler = pickle.load(open('scaler.pkl', 'rb'))
# x_train_scaled = scaler.transform(x_train)
# x_train_scaled

## Logistic Regression

In [ ]:
lr = LogisticRegression()
lr.fit(x_train_scaled, y_train)

In [ ]:
# predict for train values

y_pred_lr = lr.predict(x_train_scaled)
y_pred_lr

In [ ]:
ac_lr_tr = accuracy_score(y_pred_lr, y_train)
re_lr_tr = recall_score(y_pred_lr, y_train)
pr_lr_tr = precision_score(y_pred_lr, y_train)
f1_lr_tr = f1_score(y_pred_lr, y_train)

print("Training Accuracy: ", ac_lr_tr)
print("Training Recall Score: ", re_lr_tr)
print("Training Precision Score: ", pr_lr_tr)
print("Training F1 Score: ", f1_lr_tr)


In [ ]:
# classification report
print(classification_report(y_pred_lr,y_train))

In [ ]:
# confusion matrix
cm_lr = confusion_matrix(y_pred_lr,y_train)
cm_lr

In [ ]:
# plotting confusion matrix
cmd = ConfusionMatrixDisplay(cm_lr)
cmd.plot()

In [ ]:
# predict for test values
y_pred_lr = lr.predict(x_test_scaled)
y_pred_lr

In [ ]:
ac_lr=accuracy_score(y_pred_lr,y_test)
re_lr=recall_score(y_pred_lr,y_test)
pr_lr=precision_score(y_pred_lr,y_test)
f1_lr=f1_score(y_pred_lr,y_test)

print("Accuracy : ",ac_lr)
print("Recall Score : ",re_lr)
print("Precision Score : ",pr_lr)
print("F1 Score : ",f1_lr)

In [ ]:
# confusion matrix
cm_lr = confusion_matrix(y_pred_lr,y_test)
cm_lr

In [ ]:


# Assuming cm_lr is the confusion matrix for logistic regression
cmd = ConfusionMatrixDisplay(confusion_matrix=cm_lr)
cmd.plot(cmap='Blues')

# Add title
plt.title("Confusion Matrix - Logistic Regression")
plt.figure(dpi=300)
plt.show()

In [ ]:
# table for training test data comparison
import matplotlib.pyplot as plt
import pandas as pd

# Data for the table
metrics = ["Accuracy", "Recall", "Precision", "F1-Score"]
training_values = [ac_lr_tr, re_lr_tr, pr_lr_tr, f1_lr_tr]
test_values = [ac_lr, re_lr, pr_lr, f1_lr]

# Create a DataFrame for better visualization
data = {
    "Metrics": metrics,
    "Training Data": training_values,
    "Test Data": test_values
}
df = pd.DataFrame(data)

df = df.round(2)

# Plot the table
plt.figure(figsize=(8, 4), dpi=300)
plt.axis('tight')
plt.axis('off')
table = plt.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.auto_set_column_width(col=list(range(len(df.columns))))

plt.title("Training vs Test Data Metrics Comparison", fontsize=12)
plt.show()


## KNN

In [ ]:
# using knn classifier

knn = KNeighborsClassifier()
knn.fit(x_train_scaled, y_train)

In [ ]:
# predicting training dataset values
y_pred_knn = knn.predict(x_train_scaled)
y_pred_knn.shape

In [ ]:
# # checking accuracy, precision, f1, recall
ac_knn=accuracy_score(y_pred_knn,y_train)
re_knn=recall_score(y_pred_knn,y_train)
pr_knn=precision_score(y_pred_knn,y_train)
f1_knn=f1_score(y_pred_knn,y_train)

print("Accuracy : ",ac_knn)
print("Recall Score : ",re_knn)
print("Precision Score : ",pr_knn)
print("F1 Score : ",f1_knn)

In [ ]:
# predict for test data
y_pred_knn = knn.predict(x_test_scaled)
y_pred_knn.shape

In [ ]:
# checking accuracy, precision, f1, recall
ac_knn=accuracy_score(y_pred_knn,y_test)
re_knn=recall_score(y_pred_knn,y_test)
pr_knn=precision_score(y_pred_knn,y_test)
f1_knn=f1_score(y_pred_knn,y_test)

print("Accuracy : ",ac_knn)
print("Recall Score : ",re_knn)
print("Precision Score : ",pr_knn)
print("F1 Score : ",f1_knn)

In [ ]:
# using knn classifier

knn_list = []
knn_list2 = []

# Iterate over different values of k from 1 to 19
for i in range(1, 20):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(x_train_scaled, y_train)

    # Calculate accuracy and store in lists
    # train_accuracy = knn.score(x_train_scaled, y_train)
    # test_accuracy = knn.score(x_test_scaled, y_test)

    train_accuracy = accuracy_score(y_train, knn.predict(x_train_scaled))
    test_accuracy = accuracy_score(y_test, knn.predict(x_test_scaled))

    knn_list.append([train_accuracy, i])
    knn_list2.append([test_accuracy, i])

    # Print the results for each k
    print(f"k = {i}: Train accuracy = {train_accuracy:.4f}, Test accuracy = {test_accuracy:.4f}")

# Extracting accuracy lists for plotting
accuracy_list1 = [accuracy[0] for accuracy in knn_list]  # Train accuracies
accuracy_list2 = [accuracy[0] for accuracy in knn_list2]  # Test accuracies

print("Train accuracies:", accuracy_list1)
print("Test accuracies:", accuracy_list2)

# Plotting the accuracies
plt.figure(figsize=(10, 6))

plt.plot(range(1, 20), accuracy_list1, marker='o', label='Train Accuracy')
plt.plot(range(1, 20), accuracy_list2, marker='o', label='Test Accuracy')

plt.xlabel('K values')
plt.ylabel('Accuracy')
plt.title('KNN Accuracy for Train and Test Data')
plt.xticks(range(1, 20))
plt.legend()
plt.grid(True)
plt.show()

# Finally, retrain the KNN classifier with the best k (you can choose based on the plot)
best_k = accuracy_list2.index(max(accuracy_list2)) + 1  # +1 because k starts from 1
print(f"Best K value based on test accuracy: {best_k}")

# Retrain KNN with the best k value
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(x_train_scaled, y_train)
# knn_best.score(x_train_scaled, y_train)



In [ ]:
# predicting training dataset values
y_pred_knn = knn_best.predict(x_train_scaled)
y_pred_knn

In [ ]:
# checking accuracy, precision, f1, recall
ac_knn_tr=accuracy_score(y_pred_knn,y_train)
re_knn_tr=recall_score(y_pred_knn,y_train)
pr_knn_tr=precision_score(y_pred_knn,y_train)
f1_knn_tr=f1_score(y_pred_knn,y_train)

print("Accuracy : ",ac_knn_tr)
print("Recall Score : ",re_knn_tr)
print("Precision Score : ",pr_knn_tr)
print("F1 Score : ",f1_knn_tr)

In [ ]:
# confusion matrix
cm_lr = confusion_matrix(y_pred_knn,y_train)
cm_lr

In [ ]:
# plotting confusion matrix
cmd = ConfusionMatrixDisplay(cm_lr)
cmd.plot()

In [ ]:
# predict for test data
y_pred_knn = knn_best.predict(x_test_scaled)
y_pred_knn

In [ ]:
# checking accuracy, precision, f1, recall
ac_knn=accuracy_score(y_pred_knn,y_test)
re_knn=recall_score(y_pred_knn,y_test)
pr_knn=precision_score(y_pred_knn,y_test)
f1_knn=f1_score(y_pred_knn,y_test)

print("Accuracy : ",ac_knn)
print("Recall Score : ",re_knn)
print("Precision Score : ",pr_knn)
print("F1 Score : ",f1_knn)

In [ ]:
# confusion matrix
cm_lr = confusion_matrix(y_pred_knn,y_test)
cm_lr

In [ ]:
# plotting confusion matrix
cmd = ConfusionMatrixDisplay(cm_lr)
cmd.plot()

# Add title
plt.title("Confusion Matrix - KNN")
plt.figure(dpi=300)
plt.show()

In [ ]:
metrics = ["Accuracy", "Recall", "Precision", "F1-Score"]
training_values = [ac_knn_tr, re_knn_tr, pr_knn_tr, f1_knn_tr]
test_values = [ac_knn, re_knn, pr_knn, f1_knn]

# Create a DataFrame
data = {
    "Metrics": metrics,
    "Training Data": training_values,
    "Test Data": test_values
}
df = pd.DataFrame(data).round(2)

# Plot the table
plt.figure(figsize=(8, 4), dpi=300)
plt.axis('tight')
plt.axis('off')
table = plt.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.auto_set_column_width(col=list(range(len(df.columns))))

plt.title("Training vs Test Data Metrics Comparison (KNN)", fontsize=12)
plt.show()



## SVM

In [ ]:
# training with SVM model
svc=SVC(kernel='rbf',C=1000,gamma=0.01)
svc.fit(x_train_scaled,y_train)

In [ ]:
# predicting with training values
y_pred_svc=svc.predict(x_train_scaled)
y_pred_svc

In [ ]:
# checking accuracy, precision, f1 score, recall
ac_svc=accuracy_score(y_pred_svc,y_train)
re_svc=recall_score(y_pred_svc,y_train)
pr_svc=precision_score(y_pred_svc,y_train)
f1_svc=f1_score(y_pred_svc,y_train)

print("Accuracy : ",ac_svc)
print("Recall Score : ",re_svc)
print("Precision Score : ",pr_svc)
print("F1 Score : ",f1_svc)

In [ ]:
# confusion matrix
cm_lr = confusion_matrix(y_pred_svc,y_train)
cm_lr

In [ ]:
# plotting confusion matrix
cmd = ConfusionMatrixDisplay(cm_lr)
cmd.plot()

In [ ]:
# predicting with testing data
y_pred_svc=svc.predict(x_test_scaled)
y_pred_svc

In [ ]:
# checking accuracy, precision, f1 score, recall
ac_svc=accuracy_score(y_pred_svc,y_test)
re_svc=recall_score(y_pred_svc,y_test)
pr_svc=precision_score(y_pred_svc,y_test)
f1_svc=f1_score(y_pred_svc,y_test)

print("Accuracy : ",ac_svc)
print("Recall Score : ",re_svc)
print("Precision Score : ",pr_svc)
print("F1 Score : ",f1_svc)

In [ ]:
# confusion matrix
cm_lr = confusion_matrix(y_pred_svc,y_test)
cm_lr

In [ ]:
# plotting confusion matrix
cmd = ConfusionMatrixDisplay(cm_lr)
cmd.plot()

In [ ]:
# classification report
print(classification_report(y_pred_svc,y_test))

GridSearch on SVM

In [ ]:
#gridsearchCV on svm
from sklearn.model_selection import GridSearchCV as grid_search

C_2d_range = [1e-2, 1, 1e2]
gamma_2d_range = [1e-1, 1, 1e1]
classifiers = []
for C in C_2d_range:
    for gamma in gamma_2d_range:
        clf = SVC(C=C, gamma=gamma)
        clf.fit(x_train_scaled, y_train)
        classifiers.append((C, gamma, clf))

In [ ]:
# # predicting with training values
# y_pred_svc=grid_search.predict(x_train_scaled)
# y_pred_svc

In [ ]:
# # checking accuracy, precision, f1 score, recall
# ac_svc=accuracy_score(y_pred_svc,y_train)
# re_svc=recall_score(y_pred_svc,y_train)
# pr_svc=precision_score(y_pred_svc,y_train)
# f1_svc=f1_score(y_pred_svc,y_train)

# print("Accuracy : ",ac_svc)
# print("Recall Score : ",re_svc)
# print("Precision Score : ",pr_svc)
# print("F1 Score : ",f1_svc)

In [ ]:
# # predicting with testing data
# y_pred_svc=grid_search.predict(x_test_scaled)
# y_pred_svc

In [ ]:
# # checking accuracy, precision, f1 score, recall
# ac_svc=accuracy_score(y_pred_svc,y_test)
# re_svc=recall_score(y_pred_svc,y_test)
# pr_svc=precision_score(y_pred_svc,y_test)
# f1_svc=f1_score(y_pred_svc,y_test)

# print("Accuracy : ",ac_svc)
# print("Recall Score : ",re_svc)
# print("Precision Score : ",pr_svc)
# print("F1 Score : ",f1_svc)

## Random Forest

In [ ]:
# training with random forest classifier

classifier_rf = RandomForestClassifier(random_state=0, n_jobs=-1, max_depth=5,
                                       n_estimators=100, oob_score=True)

classifier_rf.fit(x_train_scaled, y_train)

In [ ]:
# predicting training dataset values
y_pred_rf = classifier_rf.predict(x_train_scaled)
y_pred_rf.shape

In [ ]:
# checking accuracy, precision, recall, f1 score

ac_rf = accuracy_score(y_pred_rf, y_train)
re_rf = recall_score(y_pred_rf, y_train)
pr_rf = precision_score(y_pred_rf, y_train)
f1_rf = f1_score(y_pred_rf, y_train)

print("Accuracy : ", ac_rf)
print("Recall Score : ", re_rf)
print("Precision Score : ", pr_rf)
print("F1 Score : ", f1_rf)

In [ ]:
# predicting values for testing data

y_pred_rf = classifier_rf.predict(x_test_scaled)
y_pred_rf.shape

In [ ]:
# checking accuracy, precision, recall, f1 score

ac_rf = accuracy_score(y_pred_rf, y_test)
re_rf = recall_score(y_pred_rf, y_test)
pr_rf = precision_score(y_pred_rf, y_test)
f1_rf = f1_score(y_pred_rf, y_test)

print("Accuracy : ", ac_rf)
print("Recall Score : ", re_rf)
print("Precision Score : ", pr_rf)
print("F1 Score : ", f1_rf)

Hyperparameter Tuning Using GridsearchCV

In [ ]:
rf = RandomForestClassifier(random_state=0, n_jobs=-1)

params = {
    'max_depth': [2,3,5,10,20],
    'min_samples_leaf': [5,10,20,50,100,200],
    'n_estimators': [10,25,30,50,100,200]
}


# Instantiate the grid search model
grid_search = GridSearchCV(estimator=rf,
                           param_grid=params,
                           cv = 4,
                           n_jobs=1, verbose=1, scoring="accuracy")

grid_search.fit(x_train_scaled, y_train)

In [ ]:
# checking best score
grid_search.best_score_

In [ ]:
# checking best estimator
rf_best = grid_search.best_estimator_
rf_best

In [ ]:
# checking importance of best estimator
rf_best.feature_importances_
# making a barplot for importance of all features using matplotlib
import matplotlib.pyplot as plt
import numpy as np

# Assuming `rf_best` is the trained Random Forest model
feature_names = x_train.columns
feature_importances = rf_best.feature_importances_

# Sort features by importance
indices = np.argsort(feature_importances)[::-1]
sorted_features = np.array(feature_names)[indices]
sorted_importances = feature_importances[indices]

# Plot
plt.figure(figsize=(10, 6))
plt.bar(range(len(sorted_importances)), sorted_importances, align="center")
plt.xticks(range(len(sorted_importances)), sorted_features, rotation=45, ha="right")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.title("Feature Importance")
plt.tight_layout()
plt.show()



In [ ]:
# sorting the features according to importance
imp_df = pd.DataFrame({
    "Varname": x_train.columns,
    "Imp": rf_best.feature_importances_
})

imp_df.sort_values(by="Imp", ascending=False)

In [ ]:
# from matplotlib import pyplot as plt
# _df_0['Imp'].plot(kind='hist', bins=20, title='Imp')
# plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# from matplotlib import pyplot as plt
# import seaborn as sns
# _df_1.groupby('Varname').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
# plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# plotting best estimator decision tree with class names "Diabetes" & "No Diabetes"
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10), dpi=600)
plot_tree(
    rf_best.estimators_[0],
    feature_names=x_train.columns,
    class_names=["No Diabetes", "Diabetes"],
    filled=True
)
plt.show()


In [ ]:
# predicting on training data
y_pred_rf = grid_search.predict(x_train_scaled)
y_pred_rf.shape

In [ ]:
# checking accuracy, precision, recall, f1 score

ac_rf_tr = accuracy_score(y_pred_rf, y_train)
re_rf_tr = recall_score(y_pred_rf, y_train)
pr_rf_tr = precision_score(y_pred_rf, y_train)
f1_rf_tr = f1_score(y_pred_rf, y_train)

print("Accuracy : ", ac_rf_tr)
print("Recall Score : ", re_rf_tr)
print("Precision Score : ", pr_rf_tr)
print("F1 Score : ", f1_rf_tr)

In [ ]:
# predicting on test data
y_pred_rf = grid_search.predict(x_test_scaled)
y_pred_rf.shape

In [ ]:
# checking accuracy, precision, recall, f1 score

ac_rf_best = accuracy_score(y_pred_rf, y_test)
re_rf_best = recall_score(y_pred_rf, y_test)
pr_rf_best = precision_score(y_pred_rf, y_test)
f1_rf_best = f1_score(y_pred_rf, y_test)

print("Accuracy : ", ac_rf_best)
print("Recall Score : ", re_rf_best)
print("Precision Score : ", pr_rf_best)
print("F1 Score : ", f1_rf_best)

In [ ]:
# confusion matrix
cm_rf = confusion_matrix(y_pred_rf,y_test)
cm_rf
# plotting confusion matrix
cmd = ConfusionMatrixDisplay(cm_rf )
cmd.plot()

# Add title
plt.title("Confusion Matrix - Random Forest")
plt.figure(dpi=300)
plt.show()

In [ ]:
# Data for the table
metrics = ["Accuracy", "Recall", "Precision", "F1-Score"]
training_values = [ac_rf_tr, re_rf_tr, pr_rf_tr, f1_rf_tr]
test_values = [ac_rf_best, re_rf_best, pr_rf_best, f1_rf_best]

# Create a DataFrame
data = {
    "Metrics": metrics,
    "Training Data": training_values,
    "Test Data": test_values
}
df = pd.DataFrame(data).round(2)

# Plot the table
plt.figure(figsize=(8, 4), dpi=300)
plt.axis('tight')
plt.axis('off')
table = plt.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.auto_set_column_width(col=list(range(len(df.columns))))

plt.title("Training vs Test Data Metrics Comparison (Random Forest)", fontsize=12)
plt.show()


Training Data on Best Estimator

In [ ]:
# # training with random forest classifier

# classifier_rf = RandomForestClassifier(random_state=0, n_jobs=-1, max_depth=5,
#                                        n_estimators=50, min_samples_leaf = 20)

# classifier_rf.fit(x_train_scaled, y_train)

In [ ]:
# # predicting training dataset values
# y_pred_rf = classifier_rf.predict(x_train_scaled)
# y_pred_rf.shape

In [ ]:
# # checking accuracy, precision, recall, f1 score

# ac_rf = accuracy_score(y_pred_rf, y_train)
# re_rf = recall_score(y_pred_rf, y_train)
# pr_rf = precision_score(y_pred_rf, y_train)
# f1_rf = f1_score(y_pred_rf, y_train)

# print("Accuracy : ", ac_rf)
# print("Recall Score : ", re_rf)
# print("Precision Score : ", pr_rf)
# print("F1 Score : ", f1_rf)

In [ ]:
# x_test_scaled

In [ ]:
# # predicting values for testing data

# y_pred_rf = classifier_rf.predict(x_test_scaled)
# y_pred_rf

In [ ]:
# # checking accuracy, precision, recall, f1 score

# ac_rf = accuracy_score(y_pred_rf, y_test)
# re_rf = recall_score(y_pred_rf, y_test)
# pr_rf = precision_score(y_pred_rf, y_test)
# f1_rf = f1_score(y_pred_rf, y_test)

# print("Accuracy : ", ac_rf)
# print("Recall Score : ", re_rf)
# print("Precision Score : ", pr_rf)
# print("F1 Score : ", f1_rf)

In [ ]:
# finding those unscaled values for  which model predicted 0
# x_test[y_pred_rf == 0].head(5)

In [ ]:
# x_test[y_pred_rf == 1].head(5)

In [ ]:
# # plot confusion matrix for test data
# cm_rf = confusion_matrix(y_pred_rf, y_test)
# cmd = ConfusionMatrixDisplay(cm_rf)
# cmd.plot()

## XGBoost

In [ ]:
!pip install xgboost==1.7.6 scikit-learn==1.2.2




In [ ]:
import xgboost as xgb
from xgboost import XGBClassifier

In [ ]:
# training with XGBoost classifier

xgb = XGBClassifier(random_state=0)
xgb.fit(x_train_scaled, y_train)

In [ ]:
# predicting with training data
y_pred_xgb = xgb.predict(x_train_scaled)
y_pred_xgb.shape

In [ ]:
# checking accuracy, precision, recall, f1 score
ac_xgb_tr = accuracy_score(y_pred_xgb, y_train)
re_xgb_tr = recall_score(y_pred_xgb, y_train)
pr_xgb_tr = precision_score(y_pred_xgb, y_train)
f1_xgb_tr = f1_score(y_pred_xgb, y_train)

print("Accuracy : ", ac_xgb_tr)
print("Recall Score : ", re_xgb_tr)
print("Precision Score : ", pr_xgb_tr)
print("F1 Score : ", f1_xgb_tr)

In [ ]:
# predicting for test data
y_pred_xgb = xgb.predict(x_test_scaled)
y_pred_xgb.shape

In [ ]:
# checking accuracy, precision, recall, f1 score
ac_xgb = accuracy_score(y_pred_xgb, y_test)
re_xgb = recall_score(y_pred_xgb, y_test)
pr_xgb = precision_score(y_pred_xgb, y_test)
f1_xgb = f1_score(y_pred_xgb, y_test)

print("Accuracy : ", ac_xgb)
print("Recall Score : ", re_xgb)
print("Precision Score : ", pr_xgb)
print("F1 Score : ", f1_xgb)

In [ ]:
# finding all those input for those model has predicted 0, 1

x_test[y_pred_xgb == 0].head(5)

In [ ]:
x_test[y_pred_xgb == 1].head(5)

In [ ]:
# plot confusion matrix
cm_xgb = confusion_matrix(y_pred_xgb, y_test)
cmd = ConfusionMatrixDisplay(cm_xgb)
cmd.plot()

# Add title
plt.title("Confusion Matrix - XGBoost")
plt.figure(dpi=300)
plt.show()

In [ ]:
# Data for the table
metrics = ["Accuracy", "Recall", "Precision", "F1-Score"]
training_values = [ac_xgb_tr, re_xgb_tr, pr_xgb_tr, f1_xgb_tr]
test_values = [ac_xgb, re_xgb, pr_xgb, f1_xgb]

# Create a DataFrame
data = {
    "Metrics": metrics,
    "Training Data": training_values,
    "Test Data": test_values
}
df = pd.DataFrame(data).round(2)

# Plot the table
plt.figure(figsize=(8, 4), dpi=300)
plt.axis('tight')
plt.axis('off')
table = plt.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.auto_set_column_width(col=list(range(len(df.columns))))

plt.title("Training vs Test Data Metrics Comparison (XGBoost)", fontsize=12)
plt.show()

## Decision Trees

In [ ]:
# importing decision tree classifier
from sklearn.tree import DecisionTreeClassifier

In [ ]:
# trainging with decision tree
classifier_dt = DecisionTreeClassifier(random_state=0)
classifier_dt.fit(x_train_scaled, y_train)

In [ ]:
# predicting with training data
y_pred_dt = classifier_dt.predict(x_train_scaled)
y_pred_dt.shape

In [ ]:
# checking accuracy, recall, f1 score, precision
ac_dt_tr = accuracy_score(y_pred_dt, y_train)
re_dt_tr = recall_score(y_pred_dt, y_train)
pr_dt_tr = precision_score(y_pred_dt, y_train)
f1_dt_tr = f1_score(y_pred_dt, y_train)

print("Accuracy : ", ac_dt_tr)
print("Recall Score : ", re_dt_tr)
print("Precision Score : ", pr_dt_tr)
print("F1 Score : ", f1_dt_tr)

In [ ]:
# predicting with testing data
y_pred_dt = classifier_dt.predict(x_test_scaled)
y_pred_dt.shape

In [ ]:
# checking accuracy, recall, f1 score, precision
ac_dt = accuracy_score(y_pred_dt, y_test)
re_dt = recall_score(y_pred_dt, y_test)
pr_dt = precision_score(y_pred_dt, y_test)
f1_dt = f1_score(y_pred_dt, y_test)

print("Accuracy : ", ac_dt)
print("Recall Score : ", re_dt)
print("Precision Score : ", pr_dt)
print("F1 Score : ", f1_dt)

In [ ]:
# confusion matrix
cm_dt = confusion_matrix(y_pred_dt,y_test)
cm_dt

# plotting confusion matrix
cmd = ConfusionMatrixDisplay(cm_dt )
cmd.plot()

# Add title
plt.title("Confusion Matrix - Decision Trees")
plt.figure(dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming `dt_model` is the trained Decision Tree model
feature_names = x_train.columns
feature_importances = classifier_dt.feature_importances_

# Sort features by importance
indices = np.argsort(feature_importances)[::-1]
sorted_features = np.array(feature_names)[indices]
sorted_importances = feature_importances[indices]

# Plot
plt.figure(figsize=(10, 6), dpi=300)
plt.bar(range(len(sorted_importances)), sorted_importances, align="center", color="skyblue")
plt.xticks(range(len(sorted_importances)), sorted_features, rotation=45, ha="right")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.title("Feature Importance - Decision Tree")
plt.tight_layout()
plt.show()


In [ ]:
# Data for the table
metrics = ["Accuracy", "Recall", "Precision", "F1-Score"]
training_values = [ac_dt_tr, re_dt_tr, pr_dt_tr, f1_dt_tr]
test_values = [ac_dt, re_dt, pr_dt, f1_dt]

# Create a DataFrame
data = {
    "Metrics": metrics,
    "Training Data": training_values,
    "Test Data": test_values
}
df = pd.DataFrame(data).round(2)

# Plot the table
plt.figure(figsize=(8, 4), dpi=300)
plt.axis('tight')
plt.axis('off')
table = plt.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.auto_set_column_width(col=list(range(len(df.columns))))

plt.title("Training vs Test Data Metrics Comparison (Decision Tree)", fontsize=12)
plt.show()

## Making a predictive system

In [ ]:
# predictive system
classifier = xgb   # to be updated with best performing model
scaler = pickle.load(open('scaler.pkl', 'rb'))
input = [[0,80,70,22,70.5,22,0.120,23]]
input = np.array(input)
input = scaler.transform(input)
print(input)
output = classifier.predict(input)

if output == 0:
    print("The person is not diabetic")
else:
    print("The person is diabetic")

## Saving the trained model

In [ ]:
import pickle

In [ ]:
filename = 'diabetes_model.pkl'
pickle.dump(classifier, open(filename, 'wb'))

In [ ]:
# loading the saved model
loaded_model = pickle.load(open(filename, 'rb'))
scaler = pickle.load(open('scaler.pkl', 'rb'))

In [ ]:
# predicting through loaded model

input = [[3,176,86,27,156,33.3,1.154,52]]
input = np.array(input)
input = scaler.transform(input)
print(input)
output = loaded_model.predict(input)

if output == 0:
    print("The person is not diabetic")
else:
    print("The person is diabetic")

In [ ]:
# predicting x_test on this model
input = np.array(x_test)
input = scaler.transform(input)
print(input)
output = loaded_model.predict(input)
print(output)

In [ ]:
# checking accuracy score, f1 score, recall_score, precision score
accuracy = accuracy_score(output, y_test)
f1 = f1_score(output, y_test)
recall = recall_score(output, y_test)
precision = precision_score(output, y_test)
print("Accuracy : ", accuracy)
print("F1 Score : ", f1)
print("Recall Score : ", recall)
print("Precision Score : ", precision)

## Updation with new datsets

In [ ]:
# # making a table for all models vs (accuracy, recall, F1 score, Precision score)
# models = ["Logistic Regression", "KNN", "SVM", "Random Forest", "XGBoost"]
# accuracy = [ac_lr, ac_knn, ac_svc, ac_rf, ac_xgb]
# recall = [re_lr, re_knn, re_svc, re_rf, re_xgb]
# f1 = [f1_lr, f1_knn, f1_svc, f1_rf, f1_xgb]
# precision = [pr_lr, pr_knn, pr_svc, pr_rf, pr_xgb]


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Data for the table
models = ["Logistic Regression", "KNN", "Random Forest", "XGBoost", "Decision Tree"]
accuracy = [ac_lr, ac_knn, ac_rf_best, ac_xgb, ac_dt]
recall = [re_lr, re_knn, re_rf_best, re_xgb, re_dt]
f1 = [f1_lr, f1_knn, f1_rf_best, f1_xgb, f1_dt]
precision = [pr_lr, pr_knn, pr_rf_best, pr_xgb, pr_dt]

# Create a DataFrame
results = pd.DataFrame({
    "Model": models,
    "Accuracy": accuracy,
    "Recall": recall,
    "F1 Score": f1,
    "Precision": precision
})

# Round values to 2 decimals
results = results.round(2)

# Plot the table
fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=results.values, colLabels=results.columns, loc='center', cellLoc='center')

# Style the table
table.auto_set_font_size(False)
table.set_fontsize(10)
table.auto_set_column_width(col=list(range(len(results.columns))))

plt.show()


In [ ]:
# Data for the table
models = ["Logistic Regression", "KNN", "Random Forest", "XGBoost", "Decision Tree"]
training_accuracy = [ac_lr_tr, ac_knn_tr, ac_rf_tr, ac_xgb_tr, ac_dt_tr]
testing_accuracy = [ac_lr, ac_knn, ac_rf_best, ac_xgb, ac_dt]

# Create a DataFrame
data = {
    "Models": models,
    "Training Accuracy": training_accuracy,
    "Testing Accuracy": testing_accuracy
}
df = pd.DataFrame(data).round(2)

# Plot the table
plt.figure(figsize=(8, 4), dpi=300)
plt.axis('tight')
plt.axis('off')
table = plt.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.auto_set_column_width(col=list(range(len(df.columns))))

plt.title("Training vs Testing Accuracy Comparison for all models", fontsize=12)
plt.show()